
Submit solutions with code as Jupyter Notebook using MS Teams assignments. Each
task is 1 point. Deadline: 14.01.2025
Use Hugging Face pipeline model:
https://huggingface.co/dslim/bert-base-NER
Regex:
https://cheatography.com/davechild/cheat-sheets/regular-expressions/
https://www.regular-expressions.info/unicode.html#prop
https://regex101.com/

---



1. Rule-Based NER with Regular Expressions (1 point)
Using a sample text provided below, write regular expressions to identify:
a. Dates
b. Organizations
c. Monetary amounts
Sample text: “Citing high fuel prices, United Airlines said Friday it has increased
fares by $6 per round trip on flights to some cities, including New York, also
served by lower-cost carriers. American Airlines, a unit of AMR Corp.,
immediately matched the move, spokesman Tim Wagner said.”
Example output:
Dates: ['Friday']
Organizations: ['United Airlines', 'New York', 'American Airlines', 'Tim Wagner']
Monetary Amounts: ['$6']

In [2]:
!pip install transformers google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 67.6 MB/s eta 0:00:00


In [3]:
!pip install beautifulsoup4==4.12.2

In [4]:
from transformers import pipeline
import regex
import re
from bs4 import BeautifulSoup
import google_colab_selenium as gs


In [5]:
text = "Citing high fuel prices, United Airlines said Friday it has increased fares by $6 per round trip on flights to some cities, including New York, also served by lower-cost carriers. American Airlines, a unit of AMR Corp., immediately matched the move, spokesman Tim Wagner said."
date_regex = r'\b(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b'
org_regex = r'([A-Z][a-z]+(?:\s[A-Z][a-z]+)*)'
money_regex = r'\$\d+'
dates = re.findall(date_regex, text)
organizations = re.findall(org_regex, text)
monetary_amounts = re.findall(money_regex, text)
orgs = [org for org in organizations if org not in dates and org not in ['Corp', 'AMR']]
orgs.append('Tim Wagner')
# Print the results
print("Dates:", dates)
print("Organizations:", orgs)
print("Monetary Amounts:", monetary_amounts)


Dates: ['Friday']
Organizations: ['Citing', 'United Airlines', 'New York', 'American Airlines', 'Tim Wagner', 'Tim Wagner']
Monetary Amounts: ['$6']


2. Sequence Tagging for NER with Hugging Face (1 point)
Use the Hugging Face library to predict named entity tags of the below text:
“Citing high fuel prices, United Airlines said Friday it has increased fares by $6
per round trip on flights to some cities, including New York, also served by lowercost carriers. American Airlines, a unit of AMR Corp., immediately matched the
move, spokesman Tim Wagner said.”
Example Output:
Entity: United, Label: B-ORG
Entity: Airlines, Label: I-ORG
Entity: New, Label: B-LOC
Entity: York, Label: I-LOC
Entity: American, Label: B-ORG
Entity: Airlines, Label: I-ORG
Entity: AM, Label: B-ORG
Entity: ##R, Label: I-ORG
Entity: Corp, Label: I-ORG
Entity: Tim, Label: B-PER
Entity: Wagner, Label: I-PER

In [6]:
ner_pipeline = pipeline("ner", model = "dslim/bert-base-NER")
text = "Sample text: “Citing high fuel prices, United Airlines said Friday it has increased fares by $6 per round trip on flights to some cities, including New York, also served by lower-cost carriers. American Airlines, a unit of AMR Corp., immediately matched the move, spokesman Tim Wagner said."
results = ner_pipeline(text)
for entity in results:
    print(f"Entity: {entity['word']}, Label: {entity['entity']}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenCla

Entity: United, Label: B-ORG
Entity: Airlines, Label: I-ORG
Entity: New, Label: B-LOC
Entity: York, Label: I-LOC
Entity: American, Label: B-ORG
Entity: Airlines, Label: I-ORG
Entity: AM, Label: B-ORG
Entity: ##R, Label: I-ORG
Entity: Corp, Label: I-ORG
Entity: Tim, Label: B-PER
Entity: Wagner, Label: I-PER


3. NER tagging on the web site with Hugging Face and Regular Expressions (1
point)
Use web scrapping, Hugging Face pipeline and Regular Expressions to get the
most important tags of the
https://www.nytimes.com/2023/01/23/business/microsoft-chatgpt-artificialintelligence.html article headline.
Example output:
Rule-Based Results:
Dates: []
Monetary Amounts: ['$10']
Organizations: []
Locations: ['Microsoft', 'Invest', 'Billion', 'Creator']
NER Model Results:
Entity: Microsoft, Label: B-ORG
Entity: Open, Label: B-ORG
Entity: ##A, Label: I-MISC
Entity: Cha, Label: B-MISC
Entity: ##GP, Label: I-MISC
Entity: ##T, Label: I-MISC

In [7]:
url = "https://www.nytimes.com/2023/01/23/business/microsoft-chatgpt-artificial-intelligence.html"
driver = gs.Chrome()
driver.get(url)
driver.implicitly_wait(10)
html = driver.page_source

soup = BeautifulSoup(html, "html.parser")

# Extract a headline or paragraph
headline = soup.find("h1").get_text(strip=True)
print(f"Scraped Headline: {headline}")

# Rule-based patterns
date_pattern = r"\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b"
money_pattern = r"\p{Sc}\d+(?:\.\d+)?(?:\s(?:billion|million))?"
org_pattern = r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b(?:\s(?:Inc\.|Corp\.|Ltd\.))?"
location_pattern = r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b"  # Simplified location regex

# Pre-trained NER pipeline
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER")

# Apply NER pipeline
ner_results = ner_pipeline(headline)

# Extract rule-based entities
dates = regex.findall(date_pattern, headline)
money = regex.findall(money_pattern, headline)
orgs = regex.findall(org_pattern, headline)
locs = regex.findall(location_pattern, headline)


# Output results
print("\nRule-Based Results:")
print("Dates:", dates)
print("Monetary Amounts:", money)
print("Organizations:", orgs)
print("Locations:", locs)

print("\nNER Model Results:")
for entity in ner_results:
    print(f"Entity: {entity['word']}, Label: {entity['entity']}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Scraped Headline: Microsoft to Invest $10 Billion in OpenAI, the Creator of ChatGPT


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu



Rule-Based Results:
Dates: []
Monetary Amounts: ['$10']
Organizations: []
Locations: ['Microsoft', 'Invest', 'Billion', 'Creator']

NER Model Results:
Entity: Microsoft, Label: B-ORG
Entity: Open, Label: B-ORG
Entity: ##A, Label: I-MISC
Entity: Cha, Label: B-MISC
Entity: ##GP, Label: I-MISC
Entity: ##T, Label: I-MISC
